In [1]:
from datasets import load_dataset

sedat = load_dataset("SedatAl/Turkish_Recipe", split="train")
mert = load_dataset("mertbozkurt/turkish-recipe", data_files="datav2.csv", split="train")

print(sedat)
print(mert)

/opt/anaconda3/envs/tarif/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['isim', 'malzemeler', 'tarif'],
    num_rows: 3231
})
Dataset({
    features: ['Title', 'Question', 'Category', 'Materials', 'How-to-do'],
    num_rows: 5252
})


In [2]:
import pandas as pd

def incele(ds, ad):
    df = ds.to_pandas()
    print(f"========== {ad} ==========")
    print("Satır, sütun:", df.shape)
    print("Sütunlar:", list(df.columns))

    # Kirli veri kontrolü
    print("\nBoş hücreler:\n", df.isna().sum())
    print("\nTamamen aynı satır sayısı:", df.duplicated().sum())

    # Her satırdaki tüm metin sütunlarını tek bir tarif metninde birleştir
    metin = df.astype(str).agg("\n".join, axis=1)

    # Tarif uzunlukları (karakter cinsinden)
    print("\nTarif başına karakter sayısı:")
    print(metin.str.len().describe().round(0))

    # Videodaki mantık: tüm metindeki benzersiz karakterler = vocab
    tum_metin = "".join(metin)
    karakterler = sorted(set(tum_metin))
    print("\nToplam karakter:", len(tum_metin))
    print("Benzersiz karakter (vocab boyutu):", len(karakterler))
    print(repr("".join(karakterler)))

    return df

In [3]:
df_sedat = incele(sedat, "SedatAl")
print("\n")
df_mert = incele(mert, "mertbozkurt")

========== SedatAl ==========
Satır, sütun: (3231, 3)
Sütunlar: ['isim', 'malzemeler', 'tarif']

Boş hücreler:
 isim          0
malzemeler    0
tarif         0
dtype: int64

Tamamen aynı satır sayısı: 4

Tarif başına karakter sayısı:
count    3231.0
mean     1034.0
std       422.0
min       219.0
25%       736.0
50%       964.0
75%      1246.0
max      3748.0
dtype: float64

Toplam karakter: 3341973
Benzersiz karakter (vocab boyutu): 101
'\n !"&\'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWYZabcdefghijklmnopqrstuvwxyz\xa0°½ÇÖÜâçöüğİıŞş̧̆̈–’“”…'


========== mertbozkurt ==========
Satır, sütun: (5252, 5)
Sütunlar: ['Title', 'Question', 'Category', 'Materials', 'How-to-do']

Boş hücreler:
 Title        0
Question     0
Category     0
Materials    0
How-to-do    0
dtype: int64

Tamamen aynı satır sayısı: 96

Tarif başına karakter sayısı:
count     5252.0
mean      1570.0
std        576.0
min        353.0
25%       1204.0
50%       1506.0
75%       1848.0
max      16100.0
dtype: float64

T

In [4]:
for ad, df in [("SedatAl", df_sedat), ("mertbozkurt", df_mert)]:
    print(f"========== {ad} - ilk tarif ==========")
    for sutun, deger in df.iloc[0].items():
        print(f"[{sutun}]\n{deger}\n")

========== SedatAl - ilk tarif ==========
[isim]
Fırında Tavuklu Bulgur Kapama

[malzemeler]
3 yemek kaşığı zeytinyağı, 2 yemek kaşığı tereyağı, 1 adet kuru soğan, 1 adet havuç, 1 adet kırmızı biber, 2 diş sarımsak, 1/2 adet kabak, 2 çay kaşığı tuz, 1 tatlı kaşığı kuru nane, 1 çay kaşığı toz kırmızı biber, 1 tutam karabiber, 1 yemek kaşığı domates salçası, 2 su bardağı pilavlık bulgur, 8 adet haşlanmış tavuk baget, 4 su bardağı sıcak su

[tarif]
İlk olarak soğan, havuç, biber ve kabağı ufak küpler halinde doğrayıp ayrı ayrı kenarda bekletin. Geniş bir tava ya da tencereye zeytinyağını, tereyağını koyup soğanları tencereye ekleyin ve orta ateşte hafif tuzlayarak yumuşayana kadar pişirin. Tavaya sırasıyla havuçları, biberleri, bir diş doğranmış sarımsağı ve son olarak kabakları ekleyip soteleyin. Tavaya tuzu ve baharatların ardından salçayı da ekleyin, 1 dakika kadar kavurduktan sonra bulgurları da ekleyip tüm malzeme bütünleşene kadar 3-4 dakika daha kavurun. Karışımı fırın tepsisine ak

In [5]:
import ast, re, unicodedata

# 1) mertbozkurt'u SedatAl formatına çevir
def liste_metne(s):
    try:
        return ", ".join(x.strip() for x in ast.literal_eval(s))
    except (ValueError, SyntaxError):
        return s

mert_temiz = pd.DataFrame({
    "isim": df_mert["Title"],
    "malzemeler": df_mert["Materials"].apply(liste_metne),
    "tarif": df_mert["How-to-do"],
    "kategori": df_mert["Category"],
    "kaynak": "mertbozkurt",
})
sedat_temiz = df_sedat.assign(kategori=None, kaynak="SedatAl")
df = pd.concat([sedat_temiz, mert_temiz], ignore_index=True)

# 2) Metin temizliği
TIPOGRAFI = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-", "…": "..."})

def temizle(s):
    s = unicodedata.normalize("NFC", str(s))           # s + ̧  ->  ş
    s = s.translate(TIPOGRAFI)                          # akıllı tırnak/tire -> düz
    s = s.replace("\xa0", " ")                          # görünmez boşluk -> normal boşluk
    s = re.sub(r"([.!?])(?=[A-ZÇĞİÖŞÜ])", r"\1 ", s)    # "koyun.Daha" -> "koyun. Daha"
    s = re.sub(r"\s+", " ", s)                          # çoklu boşluk -> tek boşluk
    return s.strip()

for sutun in ["isim", "malzemeler", "tarif"]:
    df[sutun] = df[sutun].apply(temizle)

# 3) Kısa ve tekrar eden tarifleri at
once = len(df)
df = df[(df["malzemeler"].str.len() > 10) & (df["tarif"].str.len() > 50)]
df = df.drop_duplicates(subset=["isim", "malzemeler"]).reset_index(drop=True)
print(f"{once} -> {len(df)} tarif")
print(df["kaynak"].value_counts())

8483 -> 6532 tarif
kaynak
mertbozkurt    3305
SedatAl        3227
Name: count, dtype: int64


In [6]:
df.to_parquet("../data/tarifler_temiz.parquet", index=False)

tum = "".join(df["isim"] + df["malzemeler"] + df["tarif"])
karakterler = sorted(set(tum))
print("Toplam karakter:", len(tum))
print("Vocab boyutu:", len(karakterler))
print(repr("".join(karakterler)))

Toplam karakter: 8155585
Vocab boyutu: 94
' !"%&\'()*+,-./0123456789:;>?ABCDEFGHIJKLMNOPQRSTUVWYZabcdefghijklmnopqrstuvwxyz°½ÇÖ×ÜâçöüğİıŞş'


In [7]:
def formatla(r):
    return (f"<TARIF_BASLA>\n"
            f"<ISIM> {r.isim}\n"
            f"<MALZEMELER> {r.malzemeler}\n"
            f"<YAPILISI> {r.tarif}\n"
            f"<TARIF_BITIR>\n")

# Tarifleri karıştır, %10'unu validation'a ayır
karisik = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(karisik) * 0.1)
val_df, train_df = karisik[:n_val], karisik[n_val:]

train_text = "".join(formatla(r) for r in train_df.itertuples())
val_text = "".join(formatla(r) for r in val_df.itertuples())

with open("../data/train.txt", "w", encoding="utf-8") as f:
    f.write(train_text)
with open("../data/val.txt", "w", encoding="utf-8") as f:
    f.write(val_text)

print(f"Train: {len(train_df)} tarif, {len(train_text):,} karakter")
print(f"Val:   {len(val_df)} tarif, {len(val_text):,} karakter")
print("\n" + train_text[:700])

Train: 5879 tarif, 7,700,317 karakter
Val:   653 tarif, 860,252 karakter

<TARIF_BASLA>
<ISIM> Sodalı Yalancı Su Böreği
<MALZEMELER> 1 şişe soda (maden suyu), 400 gr. beyaz peynir, 1 çay bardağı sıvı yağ, 1 su bardağı süt, 2 adet yumurta, 5 adet yufka
<YAPILISI> Sos malzemelerini karıştırın. Fırın kabınızı yağlayarak yufkayı bölmeden ortalayın. Her kata sos sürerek orta kata kadar çıkın. Orta katına peynir koyun. Aynı şekilde devam ederek kapatın. İlk yufkanın kenarda kalan parçalarını üstüne kapatın ve dilimleyin. Sodayı her yerine eşit olacak şekilde dökün. 180 derecede kızarana kadar 35 dakika kadar pişirin. Sodalı yalancı su böreğiniz hazır. Afiyetler!
<TARIF_BITIR>
<TARIF_BASLA>
<ISIM> Fırında Soslu Sebze
<MALZEMELER> 3 adet orta boy patates, 1/2 adet brokoli


In [13]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")
from model.char_tokenizer import CharTokenizer

tok = CharTokenizer(train_text)
print("Vocab boyutu:", tok.vocab_size)

ornek = "<ISIM> Menemen\n<MALZEMELER> 3 yumurta, 2 domates"
print(tok.decode(tok.encode(ornek)) == ornek)
print(tok.decode(tok.encode("Kuzu tandır 🍖")))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Vocab boyutu: 101
True
Kuzu tandır <UNK>


In [14]:
tok = CharTokenizer(train_text)
print(tok.vocab_size)                              # kaç olmalı?
print(tok.decode(tok.encode("Kuzu tandır 🍖")))

101
Kuzu tandır <UNK>
